In [ ]:
!date

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

In [ ]:
projdir = '/u/project/cluo/terencew/claude/project_ideas/asm_lr_hprc2'
figdir = f'{projdir}/figures/pmds'
outdir = f'{projdir}/results/qc/data/qc15'
!mkdir -p {outdir}
classes = ['never', 'rare', 'variable', 'common', 'constitutive']

### replication timing / LADs vs domain map (mechanism)

In [ ]:
freq = pd.read_csv(f'{projdir}/results/meth_bins/domain_frequency_10kb.tsv.gz', sep='\t')
rt = pd.read_csv(f'{projdir}/results/meth_bins/annotations/rt_lad_10kb.tsv.gz', sep='\t')
ann = pd.read_csv(f'{projdir}/results/qc/data/qc14/bin_annotation_variance_10kb.tsv.gz', sep='\t')
d = freq.merge(rt, on=['chrom', 'bin_start']).merge(
    ann[['chrom', 'bin_start', 'annot', 'v_between', 'v_within', 'r2_state', 'dist_const_kb']],
    on=['chrom', 'bin_start'], how='left')
d['lad'] = (d['lad_bp'] > 5000).astype(int)
d.shape

In [ ]:
d.head()

In [ ]:
t = d.groupby('class_pmd').agg(n=('rt', 'size'), rt_median=('rt', 'median'), frac_lad=('lad', 'mean'),
                               mean_mcg=('mean_meth', 'median')).reindex(classes)
t

In [ ]:
print('spearman freq_pmd vs RT:', round(d[['freq_pmd', 'rt']].corr(method='spearman').iloc[0, 1], 3))
print('spearman mean_mCG vs RT:', round(d[['mean_meth', 'rt']].corr(method='spearman').iloc[0, 1], 3))
dec = d.assign(rt_dec=pd.qcut(d['rt'], 10, labels=False)).groupby('rt_dec').agg(
    rt=('rt', 'median'), frac_constitutive=('class_pmd', lambda x: (x == 'constitutive').mean()),
    mean_mcg=('mean_meth', 'mean'), frac_lad=('lad', 'mean'), v_between=('v_between', 'median'))
dec

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
sns.boxplot(data=d, x='class_pmd', y='rt', order=classes, fliersize=0, ax=axes[0], color='lightgray')
axes[0].set_ylabel('Repli-seq wavelet (high = early)')
axes[1].plot(dec['rt'], dec['frac_constitutive'], 'o-')
axes[1].set_xlabel('replication timing (median per decile)'); axes[1].set_ylabel('fraction constitutive PMD bins')
axes[2].plot(dec['rt'], dec['mean_mcg'], 'o-', label='mean mCG')
axes[2].plot(dec['rt'], dec['frac_lad'], 's-', label='fraction LAD')
axes[2].set_xlabel('replication timing'); axes[2].legend()
for ax in axes: ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig(f'{figdir}/qc15_rt_lad_vs_domains.png', dpi=150)

### variant density (assembly-vs-hg38) by domain class, before and after adjusting for RT

In [ ]:
vd = pd.concat([pd.read_csv(f, sep='\t') for f in sorted(glob.glob(f'{projdir}/results/meth_bins/variant_density/chr*.bins.tsv.gz'))])
v = d.merge(vd.drop(columns=['n_donors_x'], errors='ignore'), on=['chrom', 'bin_start'], suffixes=('', '_vd'))
v['n_donors'] = v['n_donors_vd'] if 'n_donors_vd' in v else v['n_donors']
ncpg = pd.concat([pd.read_csv(f, sep='\t', usecols=['chrom', 'bin_start', 'n_cpg'])
                  for f in glob.glob(f'{projdir}/results/meth_bins/per_hap/HG00097_hap1.bins10kb.tsv.gz')])
v = v.merge(ncpg, on=['chrom', 'bin_start'], how='left')
for c in ['snv', 'indel', 'sv']:
    v[f'{c}_pd'] = v[c] / v['n_donors']
v['log_ncpg'] = np.log10(v['n_cpg'].fillna(0) + 1)
v.groupby('class_pmd')[['snv_pd', 'indel_pd', 'sv_pd']].median().reindex(classes).round(3)

In [ ]:
rows = []
for y in ['snv_pd', 'indel_pd', 'sv_pd']:
    x = v.dropna(subset=[y, 'rt', 'log_ncpg'])
    m0 = smf.ols(f'{y} ~ C(class_pmd)', x).fit()
    m1 = smf.ols(f'{y} ~ C(class_pmd) + log_ncpg + gene_bp', x) if 'gene_bp' in x else None
    m2 = smf.ols(f'{y} ~ C(class_pmd) + log_ncpg + rt', x).fit()
    m3 = smf.ols(f'{y} ~ rt + log_ncpg', x).fit()
    rows.append({'metric': y, 'r2_class': m0.rsquared, 'r2_class_cpg_rt': m2.rsquared, 'r2_rt_cpg': m3.rsquared,
                 'coef_constitutive_unadj': m0.params.get('C(class_pmd)[T.constitutive]', np.nan),
                 'coef_constitutive_rt_adj': m2.params.get('C(class_pmd)[T.constitutive]', np.nan)})
vt = pd.DataFrame(rows)
vt

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, y in zip(axes, ['snv_pd', 'indel_pd', 'sv_pd']):
    sns.boxplot(data=v, x='class_pmd', y=y, order=classes, fliersize=0, showfliers=False, ax=ax, color='lightgray')
    ax.set_title(y, fontsize=9); ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.savefig(f'{figdir}/qc15_variant_density_by_class.png', dpi=150)

### solo-WCGW mitotic clock

In [ ]:
sw = pd.concat([pd.read_csv(f, sep='\t') for f in glob.glob(f'{projdir}/results/meth_bins/solo_wcgw/per_hap/*.tsv')])
sw_w = sw.pivot_table(index=['sample', 'hap'], columns=['site_set', 'class_pmd'], values='mean_mcg')
sw_w.columns = [f'{a}_{b}' for a, b in sw_w.columns]
sw_w = sw_w.reset_index()
sw_w['solo_depth'] = sw_w['solo_never'] - sw_w['solo_constitutive']
sw_w['all_depth'] = sw_w['all_never'] - sw_w['all_constitutive']
print(len(sw_w), 'haplotypes')
sw_w[['solo_constitutive', 'all_constitutive', 'solo_never', 'all_never', 'solo_depth', 'all_depth']].describe().round(2)

In [ ]:
chem = pd.read_csv(f'{projdir}/results/qc/data/supp_seq_qc.csv').set_index('sample_id')['sequencing_chemistry_ont']
sw_w['chemistry'] = sw_w['sample'].map(chem)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
sns.scatterplot(data=sw_w, x='all_constitutive', y='solo_constitutive', hue='chemistry', s=14, ax=axes[0])
axes[0].set_xlabel('mCG in constitutive domains, all CpGs (%)'); axes[0].set_ylabel('solo-WCGW (%)')
sns.scatterplot(data=sw_w, x='all_depth', y='solo_depth', hue='chemistry', s=14, ax=axes[1], legend=False)
axes[1].set_xlabel('domain depth, all CpGs'); axes[1].set_ylabel('domain depth, solo-WCGW')
sns.histplot(data=sw_w.melt(id_vars='chemistry', value_vars=['solo_constitutive', 'all_constitutive']),
             x='value', hue='variable', bins=40, ax=axes[2])
axes[2].set_xlabel('mCG in constitutive domains (%)')
plt.tight_layout(); plt.savefig(f'{figdir}/qc15_solo_wcgw.png', dpi=150)
print('corr(all_depth, solo_depth):', round(sw_w[['all_depth', 'solo_depth']].corr().iloc[0, 1], 3))
print('CV all:', round(sw_w['all_constitutive'].std() / sw_w['all_constitutive'].mean(), 3),
      ' CV solo:', round(sw_w['solo_constitutive'].std() / sw_w['solo_constitutive'].mean(), 3))

In [ ]:
for y in ['all_constitutive', 'solo_constitutive', 'all_depth', 'solo_depth']:
    m = smf.ols(f'{y} ~ C(chemistry)', sw_w.dropna(subset=['chemistry'])).fit()
    print(f'{y}: chemistry R2 {m.rsquared:.3f}')

### residual outside-domain axis vs clonality (XIST) and RNA state

In [ ]:
z = np.load(f'{projdir}/results/meth_bins/donor_bin_meth_10kb.npz', allow_pickle=True)
meth, donors = z['meth'], list(z['donors'])
dm = pd.read_csv(f'{projdir}/results/qc/data/qc13/donor_pmd_expansion_metrics.tsv', sep='\t').set_index('sample').loc[donors].reset_index()
idx_never = np.flatnonzero(freq['class_pmd'].values == 'never')
X = np.column_stack([np.ones(len(donors)), dm['depth_constitutive'], (dm['chemistry'] == 'R1041').astype(float)])
Y = meth[:, idx_never]
ok = ~np.isnan(Y).any(axis=0)
Y = Y[:, ok]
beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
R = Y - X @ beta
u, s, vt_ = np.linalg.svd(R - R.mean(axis=0), full_matrices=False)
var = s ** 2 / (s ** 2).sum()
pcs = pd.DataFrame(u[:, :5] * s[:5], columns=[f'rPC{i+1}' for i in range(5)])
pcs['sample'] = donors
print('residual variance explained:', var[:5].round(3))

In [ ]:
xist = pd.read_csv(f'{projdir}/results/qc/data/xist_promoter_skew.tsv', sep='\t')
rna = pd.read_csv(f'{projdir}/results/qc/data/rna_markers_wide.tsv', sep='\t')
sets = {'proliferation': ['MKI67', 'TOP2A', 'PCNA', 'CCNB1', 'AURKB'],
        'plasmablast': ['PRDM1', 'XBP1', 'IRF4', 'CD38'],
        'naive_memory_B': ['TCL1A', 'CD27', 'MS4A1', 'CD19', 'CR2'],
        'activation': ['CD40', 'NFKB1', 'TRAF1', 'ICAM1']}
for k, g in sets.items():
    have = [x for x in g if x in rna.columns]
    rna[k] = np.log1p(rna[have]).mean(axis=1)
m = pcs.merge(dm, on='sample').merge(xist[['sample', 'skew', 'hap1_meth', 'hap2_meth']], on='sample', how='left').merge(
    rna[['sample'] + list(sets)], on='sample', how='left')
m.head()

In [ ]:
rows = []
for p in [f'rPC{i+1}' for i in range(5)] + ['depth_constitutive', 'mcg_never']:
    for c in ['skew'] + list(sets):
        x = m.dropna(subset=[p, c])
        if len(x) < 20:
            continue
        r = smf.ols(f'{p} ~ {c}', x).fit()
        rows.append({'y': p, 'covariate': c, 'n': len(x), 'r2': r.rsquared, 'p': r.f_pvalue})
res = pd.DataFrame(rows)
res.pivot(index='covariate', columns='y', values='r2').round(3)

In [ ]:
res.pivot(index='covariate', columns='y', values='p').round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
sns.histplot(data=m.dropna(subset=['skew']), x='skew', bins=30, ax=axes[0])
axes[0].set_xlabel('XIST promoter skew |hap1-hap2| (clonality)')
sns.scatterplot(data=m, x='skew', y='depth_constitutive', hue='chemistry', s=16, ax=axes[1])
sns.scatterplot(data=m, x='proliferation', y='depth_constitutive', hue='chemistry', s=16, ax=axes[2], legend=False)
plt.tight_layout(); plt.savefig(f'{figdir}/qc15_clonality_rna_vs_domains.png', dpi=150)

### write outputs

In [ ]:
t.to_csv(f'{outdir}/rt_lad_by_domain_class.tsv', sep='\t')
dec.to_csv(f'{outdir}/domains_by_rt_decile.tsv', sep='\t')
vt.to_csv(f'{outdir}/variant_density_models.tsv', sep='\t', index=False)
v.groupby('class_pmd')[['snv_pd', 'indel_pd', 'sv_pd']].median().reindex(classes).to_csv(f'{outdir}/variant_density_by_class.tsv', sep='\t')
sw_w.to_csv(f'{outdir}/solo_wcgw_per_hap.tsv', sep='\t', index=False)
res.to_csv(f'{outdir}/residual_axis_vs_clonality_rna.tsv', sep='\t', index=False)
m.to_csv(f'{outdir}/donor_residual_pcs_with_covariates.tsv', sep='\t', index=False)

In [ ]:
!date